# 90 GB VRAM Causal RoPE Transformer (BPE + Project Gutenberg Edition)
**Architectural & Pipeline Upgrades:**
- **Precision & XLA Fusion:** `mixed_bfloat16` + `jit_compile=True` for Tensor Core throughput without FP16 underflow.
- **Custom Byte-Level BPE Tokenizer (`16,384` Vocab):** Trained in ~2s via Rust-backed Hugging Face `tokenizers` with **zero `<UNK>` tokens**.
- **Multi-Author Project Gutenberg Corpus (~9.5 MB):** Complete Shakespeare, Milton (*Paradise Lost* & *Paradise Regained*), Marlowe (*Doctor Faustus* & *Tamburlaine*), Austen (*Pride & Prejudice* & *Sense & Sensibility*), and Shelley (*Frankenstein*).
- **Pre-RMSNorm + SwiGLU + Weight Tying:** Eliminates Post-LN rank collapse and saves ~12.5M redundant embedding parameters.
- **Zero-Retrace GPU Sampling Callback:** Uses a fixed-shape `(1, SEQ_LENGTH)` rolling buffer so XLA compiles inference once.

In [ ]:
!pip install -q tokenizers

import os
import re
import urllib.request
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import tensorflow as tf
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

# Enable TensorFlow Memory Growth immediately at startup
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled — TF will now take only what it needs.")
    except RuntimeError as e:
        print("Memory growth already set or cannot be modified:", e)

# Unlock Tensor Cores with bfloat16 (prevents gradient underflow on datacenter GPUs)
tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')

# ==========================================================
# ⚙️ 90 GB VRAM GLOBAL CONFIGURATION & CONTROL PANEL
# ==========================================================
VOCAB_SIZE   = 16384     # Custom Byte-Level BPE vocabulary size (multiple of 128 for Tensor Cores)
SEQ_LENGTH   = 4096      # Context window (set to 2048 or 4096)
STRIDE       = 256       # Sliding window step across the corpus
BATCH_SIZE   = 32        # High-VRAM batch size (drop to 32 if using SEQ_LENGTH = 4096)
EMBED_DIM    = 768       # Hidden dimension
NUM_HEADS    = 6        # 64 dimensions per attention head
FF_DIM       = 2048      # SwiGLU intermediate dimension (~8/3 * EMBED_DIM)
NUM_LAYERS   = 6        # Transformer decoder depth
DROPOUT      = 0.10      # Residual & attention dropout

# Generation Sampling Controls
GEN_LENGTH   = 2400       # BPE subword tokens to generate per callback
TEMPERATURE  = 1.2      # 0.8 - 1.0 = coherent & creative; >1.2 = fever dream
TOP_P        = 0.96      # Nucleus sampling threshold
TOP_K        = 128        # Top-K cutoff
PRINT_FREQ   = 5         # Print sample every N epochs
# ==========================================================

print('TensorFlow Version:', tf.__version__)
print('Active Mixed Precision Policy:', tf.keras.mixed_precision.global_policy().name)
print('Detected GPUs:', tf.config.list_physical_devices('GPU'))

Memory growth enabled — TF will now take only what it needs.
TensorFlow Version: 2.21.0
Active Mixed Precision Policy: mixed_bfloat16
Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ==========================================================
# 📚 PROJECT GUTENBERG MULTI-BOOK DOWNLOADER & CLEANER
# ==========================================================
GUTENBERG_BOOKS = {
    'Shakespeare_Complete_Works': 'https://www.gutenberg.org/cache/epub/100/pg100.txt',
    'Milton_Paradise_Lost':       'https://www.gutenberg.org/cache/epub/26/pg26.txt',
    'Milton_Paradise_Regained':   'https://www.gutenberg.org/cache/epub/58/pg58.txt',
    'Marlowe_Doctor_Faustus':     'https://www.gutenberg.org/cache/epub/779/pg779.txt',
    'Marlowe_Tamburlaine':        'https://www.gutenberg.org/cache/epub/1094/pg1094.txt',
    'Austen_Pride_and_Prejudice': 'https://www.gutenberg.org/cache/epub/1342/pg1342.txt',
    'Austen_Sense_Sensibility':   'https://www.gutenberg.org/cache/epub/161/pg161.txt',
    'Shelley_Frankenstein':       'https://www.gutenberg.org/cache/epub/84/pg84.txt',
}

os.makedirs('gutenberg_corpus', exist_ok=True)

def strip_gutenberg_boilerplate(text):
    start_match = re.search(r"\*\*\*\s*START OF (THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE | re.DOTALL)
    end_match = re.search(r"\*\*\*\s*END OF (THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE | re.DOTALL)
    start_idx = start_match.end() if start_match else 0
    end_idx = end_match.start() if end_match else len(text)
    cleaned = text[start_idx:end_idx].strip()
    # Normalize Windows CRLF line endings and excessive blank lines
    cleaned = cleaned.replace('\r\n', '\n').replace('\r', '\n')
    cleaned = re.sub(r'\n{4,}', '\n\n\n', cleaned)
    return cleaned

corpus_documents = []
for name, url in GUTENBERG_BOOKS.items():
    file_path = os.path.join('gutenberg_corpus', f'{name}.txt')
    if not os.path.exists(file_path):
        print(f'Fetching {name}...')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            raw_bytes = response.read()
        with open(file_path, 'wb') as f:
            f.write(raw_bytes)
    with open(file_path, 'r', encoding='utf-8-sig', errors='ignore') as f:
        cleaned_doc = strip_gutenberg_boilerplate(f.read())
        corpus_documents.append(cleaned_doc)
        print(f'  -> {name:<28}: {len(cleaned_doc):>9,} chars')

EOS_TOKEN = '<|endoftext|>'
PAD_TOKEN = '<|pad|>'
full_corpus_text = f'\n\n{EOS_TOKEN}\n\n'.join(corpus_documents)

corpus_file = 'gutenberg_corpus/combined_cleaned_corpus.txt'
with open(corpus_file, 'w', encoding='utf-8') as f:
    f.write(full_corpus_text)

print(f'\nTotal Cleaned Literary Corpus: {len(full_corpus_text):,} characters ({len(full_corpus_text) / (1024**2):.2f} MB)')

  -> Shakespeare_Complete_Works  : 5,358,683 chars
  -> Milton_Paradise_Lost        :   459,507 chars
  -> Milton_Paradise_Regained    :    90,678 chars
  -> Marlowe_Doctor_Faustus      :   124,312 chars
  -> Marlowe_Tamburlaine         :   160,166 chars
  -> Austen_Pride_and_Prejudice  :   728,457 chars
  -> Austen_Sense_Sensibility    :   670,572 chars
  -> Shelley_Frankenstein        :   419,270 chars

Total Cleaned Literary Corpus: 8,011,764 characters (7.64 MB)


In [ ]:
# ==========================================================
# 🔤 RUST BYTE-LEVEL BPE TOKENIZER & STRIDED DATASET PIPELINE
# ==========================================================
print(f'Training custom Byte-Level BPE Tokenizer (target vocab: {VOCAB_SIZE})...')

bpe_tokenizer = Tokenizer(models.BPE())
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=[PAD_TOKEN, EOS_TOKEN]
)

bpe_tokenizer.train([corpus_file], trainer=trainer)
bpe_tokenizer.save('gutenberg_bpe_tokenizer.json')
actual_vocab_size = bpe_tokenizer.get_vocab_size()
pad_id = bpe_tokenizer.token_to_id(PAD_TOKEN)
eos_id = bpe_tokenizer.token_to_id(EOS_TOKEN)

print('Encoding full corpus into BPE token IDs...')
encoded_corpus = bpe_tokenizer.encode(full_corpus_text)
tokens_as_int = np.array(encoded_corpus.ids, dtype=np.int32)

# Zero-copy strided sliding windows via NumPy
all_windows = sliding_window_view(tokens_as_int, SEQ_LENGTH + 1)[::STRIDE]
# Copy into a contiguous array for fast Tensor slicing
all_windows = np.array(all_windows, dtype=np.int32, copy=True)

# Shuffle sequences before batching
rng = np.random.default_rng(seed=42)
rng.shuffle(all_windows)

def split_input_target(chunk):
    return chunk[:, :-1], chunk[:, 1:]

dataset = (
    tf.data.Dataset.from_tensor_slices(all_windows)
    .shuffle(10000, reshuffle_each_iteration=True)
    .repeat()
    .batch(BATCH_SIZE, drop_remainder=True)
    .map(split_input_target, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

STEPS_PER_EPOCH = len(all_windows) // BATCH_SIZE
compression_ratio = len(full_corpus_text) / len(tokens_as_int)

print(f'Active BPE Vocabulary Size : {actual_vocab_size:,}')
print(f'Total Corpus BPE Tokens    : {len(tokens_as_int):,} ({compression_ratio:.2f} chars/token)')
print(f'Strided Training Sequences : {len(all_windows):,} (length {SEQ_LENGTH})')
print(f'Steps Per Epoch            : {STEPS_PER_EPOCH:,} (Batch Size: {BATCH_SIZE})')

Training custom Byte-Level BPE Tokenizer (target vocab: 16384)...
Encoding full corpus into BPE token IDs...
Active BPE Vocabulary Size : 16,384
Total Corpus BPE Tokens    : 2,189,232 (3.66 chars/token)
Strided Training Sequences : 8,536 (length 4096)
Steps Per Epoch            : 266 (Batch Size: 32)


In [ ]:
# ==========================================================
# 🧠 PRE-RMSNORM + CACHED ROPE + SWIGLU + TIED WEIGHTS MODEL
# ==========================================================
class RMSNorm(tf.keras.layers.Layer):
    def __init__(self, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def build(self, input_shape):
        self.scale = self.add_weight(
            name='scale',
            shape=(input_shape[-1],),
            initializer='ones',
            trainable=True
        )

    def call(self, x):
        x_fp32 = tf.cast(x, tf.float32)
        normed = x_fp32 * tf.math.rsqrt(tf.reduce_mean(tf.square(x_fp32), axis=-1, keepdims=True) + self.eps)
        return tf.cast(normed, x.dtype) * self.scale


class CausalSelfAttentionWithRoPE(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, max_seq_len=4096, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Fused QKV projection for single-kernel GEMM
        self.qkv_proj = tf.keras.layers.Dense(3 * embed_dim, use_bias=False)
        self.out_proj = tf.keras.layers.Dense(embed_dim, use_bias=False)
        self.attn_dropout = tf.keras.layers.Dropout(dropout)

        # Precompute Rotary Positional Embedding tables once
        inv_freq = 1.0 / (10000.0 ** (np.arange(0, self.head_dim, 2, dtype=np.float32) / self.head_dim))
        t = np.arange(max_seq_len, dtype=np.float32)
        freqs = np.outer(t, inv_freq)
        emb = np.concatenate([freqs, freqs], axis=-1)
        self.cos_cached = tf.constant(np.cos(emb)[None, None, :, :], dtype=tf.float32)
        self.sin_cached = tf.constant(np.sin(emb)[None, None, :, :], dtype=tf.float32)

    def _apply_rope(self, x, seq_len):
        cos = tf.cast(self.cos_cached[:, :, :seq_len, :], x.dtype)
        sin = tf.cast(self.sin_cached[:, :, :seq_len, :], x.dtype)
        x1 = x[..., :self.head_dim // 2]
        x2 = x[..., self.head_dim // 2:]
        rotate_x = tf.concat([-x2, x1], axis=-1)
        return (x * cos) + (rotate_x * sin)

    def call(self, x, training=False):
        batch_size = tf.shape(x)[0]
        seq_len = tf.shape(x)[1]

        qkv = self.qkv_proj(x)
        qkv = tf.reshape(qkv, (batch_size, seq_len, 3, self.num_heads, self.head_dim))
        qkv = tf.transpose(qkv, [2, 0, 3, 1, 4])
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self._apply_rope(q, seq_len)
        k = self._apply_rope(k, seq_len)

        scale = tf.cast(tf.math.rsqrt(tf.cast(self.head_dim, tf.float32)), q.dtype)
        scores = tf.matmul(q, k, transpose_b=True) * scale

        mask = tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0)
        neg_inf = tf.cast(-1e4 if scores.dtype in (tf.bfloat16, tf.float16) else -1e9, scores.dtype)
        scores = tf.where(mask[tf.newaxis, tf.newaxis, :, :], scores, neg_inf)

        weights = tf.cast(tf.nn.softmax(tf.cast(scores, tf.float32), axis=-1), x.dtype)
        weights = self.attn_dropout(weights, training=training)

        attn_out = tf.matmul(weights, v)
        attn_out = tf.reshape(tf.transpose(attn_out, [0, 2, 1, 3]), (batch_size, seq_len, self.embed_dim))
        return self.out_proj(attn_out)


class SwiGLUFFN(tf.keras.layers.Layer):
    def __init__(self, embed_dim, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.w_gate_up = tf.keras.layers.Dense(2 * ff_dim, use_bias=False)
        self.w_down = tf.keras.layers.Dense(embed_dim, use_bias=False)
        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        gate_up = self.w_gate_up(x)
        gate, up = tf.split(gate_up, num_or_size_splits=2, axis=-1)
        hidden = tf.nn.silu(gate) * up
        return self.dropout(self.w_down(hidden), training=training)


class TransformerRoPEBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, max_seq_len=4096, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = RMSNorm()
        self.attn = CausalSelfAttentionWithRoPE(embed_dim, num_heads, max_seq_len=max_seq_len, dropout=dropout)
        self.dropout1 = tf.keras.layers.Dropout(dropout)

        self.norm2 = RMSNorm()
        self.ffn = SwiGLUFFN(embed_dim, ff_dim, dropout=dropout)

    def call(self, x, training=False):
        x = x + self.dropout1(self.attn(self.norm1(x), training=training), training=training)
        x = x + self.ffn(self.norm2(x), training=training)
        return x


class TiedEmbeddingOutput(tf.keras.layers.Layer):
    def __init__(self, embedding_layer, **kwargs):
        super().__init__(dtype='float32', **kwargs)
        self.embedding_layer = embedding_layer

    def call(self, x):
        x_fp32 = tf.cast(x, tf.float32)
        weights_fp32 = tf.cast(self.embedding_layer.embeddings, tf.float32)
        return tf.matmul(x_fp32, weights_fp32, transpose_b=True)


def build_model(vocab_size, seq_len=SEQ_LENGTH):
    inputs = tf.keras.Input(shape=(seq_len,), dtype='int32')
    tok_emb = tf.keras.layers.Embedding(vocab_size, EMBED_DIM, name='bpe_embedding')
    x = tok_emb(inputs)

    for i in range(NUM_LAYERS):
        x = TransformerRoPEBlock(
            EMBED_DIM, NUM_HEADS, FF_DIM, max_seq_len=seq_len, dropout=DROPOUT, name=f'rope_block_{i}'
        )(x)

    x = RMSNorm(name='final_rmsnorm')(x)
    outputs = TiedEmbeddingOutput(tok_emb, name='tied_lm_head')(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name='Gutenberg_RoPE_Transformer')

model = build_model(actual_vocab_size, seq_len=SEQ_LENGTH)
model.summary()

Model: "Gutenberg_RoPE_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bpe_embedding (Embedding)       │ (None, 4096, 768)      │    12,582,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rope_block_0                    │ (None, 4096, 768)      │     7,079,424 │
│ (TransformerRoPEBlock)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rope_block_1                    │ (None, 4096, 768)      │     7,079,424 │
│ (TransformerRoPEBlock)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rope_block_2                    │ (None, 4096, 768)      │     7,079,424 │
│ (TransformerRoPEBlock)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rope_block_3                    │ (None, 4096, 768)      │     7,079,424 │
│ (TransformerRoPEBlock)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rope_block_4                    │ (None, 4096, 768)      │     7,079,424 │
│ (TransformerRoPEBlock)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rope_block_5                    │ (None, 4096, 768)      │     7,079,424 │
│ (TransformerRoPEBlock)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_rmsnorm (RMSNorm)         │ (None, 4096, 768)      │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tied_lm_head                    │ (None, 4096, 16384)    │    12,582,912 │
│ (TiedEmbeddingOutput)           │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,060,224 (210.04 MB)

 Trainable params: 55,060,224 (210.04 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ==========================================================
# ⚡ ZERO-RETRACE XLA GPU SAMPLING ENGINE & CALLBACK
# ==========================================================
class FastGPUSamplingCallback(tf.keras.callbacks.Callback):
    def __init__(
        self,
        tokenizer,
        start_text="ROMEO:\n",
        gen_length=GEN_LENGTH,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        interval=PRINT_FREQ
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.start_text = start_text
        self.gen_length = gen_length
        self.temperature = temperature
        self.top_p = top_p
        self.top_k = top_k if (top_k is not None and top_k > 0) else actual_vocab_size
        self.interval = interval

    @tf.function(jit_compile=True)
    def _generate_loop(self, buffer, cur_len, gen_len, temp, top_p, top_k):
        generated = tf.TensorArray(tf.int32, size=gen_len, dynamic_size=False)

        for i in tf.range(gen_len):
            read_idx = cur_len - 1

            # Forward pass
            predictions = self.model(buffer, training=False)
            logits = tf.cast(predictions[0, read_idx, :], tf.float32) / tf.maximum(temp, 1e-5)

            # Top-K
            top_k_logits, top_k_indices = tf.math.top_k(logits, k=top_k)
            probs = tf.nn.softmax(top_k_logits)

            # Top-P (Nucleus)
            cum_probs = tf.cumsum(probs)
            nucleus_mask = (cum_probs - probs) > top_p
            filtered_logits = tf.where(nucleus_mask, tf.fill(tf.shape(top_k_logits), -1e9), top_k_logits)

            # Sample
            sampled_idx = tf.random.categorical(tf.expand_dims(filtered_logits, 0), num_samples=1, dtype=tf.int32)[0, 0]
            next_id = top_k_indices[sampled_idx]

            generated = generated.write(i, next_id)

            # Conditionally update the buffer
            buffer = tf.cond(
                cur_len < SEQ_LENGTH,
                lambda: tf.tensor_scatter_nd_update(buffer, [[0, cur_len]], [next_id]),
                lambda: tf.concat([buffer[:, 1:], [[next_id]]], axis=1)
            )

            cur_len = tf.minimum(cur_len + 1, SEQ_LENGTH)

        return generated.stack()

    def generate_text(self, prompt_text, gen_length=None, temperature=None, top_p=None, top_k=None):
        gen_len = gen_length or self.gen_length
        temp_t = tf.constant(temperature if temperature is not None else self.temperature, dtype=tf.float32)
        top_p_t = tf.constant(top_p if top_p is not None else self.top_p, dtype=tf.float32)
        top_k_t = tf.constant(top_k if top_k is not None else self.top_k, dtype=tf.int32)
        gen_len_t = tf.constant(gen_len, dtype=tf.int32)

        prompt_ids = self.tokenizer.encode(prompt_text).ids
        if not prompt_ids:
            prompt_ids = [eos_id]

        buffer_np = np.full((1, SEQ_LENGTH), pad_id, dtype=np.int32)
        init_slice = prompt_ids[-SEQ_LENGTH:]
        cur_len_val = len(init_slice)
        buffer_np[0, :cur_len_val] = init_slice

        buffer_tf = tf.constant(buffer_np, dtype=tf.int32)
        cur_len_t = tf.constant(cur_len_val, dtype=tf.int32)

        # Run the entire generation loop on GPU
        new_tokens = self._generate_loop(buffer_tf, cur_len_t, gen_len_t, temp_t, top_p_t, top_k_t)

        # Bring the final tokens back to CPU
        generated_ids = list(prompt_ids) + new_tokens.numpy().tolist()
        return self.tokenizer.decode(generated_ids)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.interval != 0 and epoch != 0:
            return

        output_str = self.generate_text(self.start_text)
        loss_val = logs.get('loss', 0.0) if logs else 0.0
        perplexity = np.exp(min(loss_val, 20.0))

        print(f"\n=== Sample Output after Epoch {epoch + 1} | Loss: {loss_val:.4f} | PPL: {perplexity:.2f} ===")
        print(f"Params -> Temp: {self.temperature} | Top_P: {self.top_p} | Top_K: {self.top_k}")
        print(output_str)
        print('=' * 75 + '\n')

In [ ]:
# ==========================================================
# 🚀 TRAINING EXECUTION (LINEAR WARMUP + COSINE DECAY + XLA)
# ==========================================================
EPOCHS = 600
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS
WARMUP_STEPS = min(500, max(50, TOTAL_STEPS // 10))

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=max(1, TOTAL_STEPS - WARMUP_STEPS),
    alpha=0.05,
    warmup_target=6e-4,
    warmup_steps=WARMUP_STEPS
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=1e-2,
    beta_1=0.9,
    beta_2=0.95,
    clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    jit_compile=True
)

sample_callback = FastGPUSamplingCallback(
    tokenizer=bpe_tokenizer,
    start_text="Of Man's first disobedience, and the fruit\n"
)

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='gutenberg_rope_best.weights.h5',
    save_weights_only=True,
    monitor='loss',
    mode='min',
    save_best_only=True
)

print(f'Launching 90 GB VRAM XLA-compiled run ({STEPS_PER_EPOCH} steps/epoch, {WARMUP_STEPS} warmup steps)...')
history = model.fit(
    dataset,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=[sample_callback, checkpoint_callback]
)

Launching 90 GB VRAM XLA-compiled run (266 steps/epoch, 500 warmup steps)...
Epoch 1/600
266/266 ━━━━━━━━━━━━━━━━━━━━ 0s 859ms/step - loss: 6.8862
=== Sample Output after Epoch 1 | Loss: 5.7657 | PPL: 319.18 ===
Params -> Temp: 1.2 | Top_P: 0.96 | Top_K: 128
Of Man's first disobedience, and the fruit
Cumbge, and with at H. The oldness
In the first tree the place through
Each early, in the part of the day,
Bourid, which were in the top.

With his head and two days; but so long
From a manner, first not above.
But let me make to try and in the world,
His own looks, and like half his friends alone,
Or being even down to his part doth reign,
Nor all her of such days which hath thought,
While ’gainst two hand I now found,
As I with one was all of my conscience,
And so I looked so by in silence to hear
To be full of that he, for good day;
Our sister did I deny her eyes my own will,
And would we stay of this than that by this present
Even in him a thousand and just. And here shall have,
As onc

KeyboardInterrupt: 

In [ ]:
# ==========================================================
# 🎮 INTERACTIVE GENERATION PLAYGROUND (RUN ANYTIME)
# ==========================================================
# Tweak prompts and sampling parameters on the fly without retraining!
custom_prompts = [
    "HAMLET:\nTo be, or not to be, ",
    "Satan, with vast extended wings, beheld\n",
    "It is a truth universally acknowledged, that a single man "
]

for prompt in custom_prompts:
    print('PROMPT:', repr(prompt))
    print('-' * 60)
    print(sample_callback.generate_text(
        prompt_text=prompt,
        gen_length=300,
        temperature=0.80,
        top_p=0.90,
        top_k=50
    ))
    print('=' * 75 + '\n')